# Chapter 10 — Distributed Data Parallel Training

Training GPT-3 (175B parameters) on a single GPU would take decades. Modern LLM training requires distributing computation across hundreds or thousands of GPUs working in concert. This chapter covers the parallelism strategies that make large-scale training possible, from the straightforward to the highly sophisticated.

### Glossary / Glossário

• DDP — Distributed Data Parallel, PyTorch strategy that replicates the model across GPUs. / Distributed Data Parallel, estratégia PyTorch que replica o modelo entre GPUs.
• data parallelism — splitting batches across GPUs, each holding a full model copy. / Dividir batches entre GPUs, cada uma com cópia completa do modelo.
• All-Reduce — collective operation that sums gradients across all GPUs and broadcasts the result. / Operação coletiva que soma gradientes de todas as GPUs e distribui o resultado.
• ZeRO — Zero Redundancy Optimizer, shards optimizer states/gradients/parameters across GPUs to save memory. / Zero Redundancy Optimizer, fragmenta estados do otimizador/gradientes/parâmetros entre GPUs para economizar memória.
• pipeline parallelism — splitting model layers across GPUs in sequential stages. / Dividir camadas do modelo entre GPUs em estágios sequenciais.
• tensor parallelism — splitting individual layers across GPUs for intra-layer distribution. / Dividir camadas individuais entre GPUs para distribuição intra-camada.
• NCCL — NVIDIA Collective Communication Library for efficient multi-GPU communication. / Biblioteca de comunicação coletiva da NVIDIA para comunicação multi-GPU eficiente.

In [ ]:
import torch
import torch.nn as nn

world_size = 4  # number of GPUs in the cluster (each processes a different data shard)
batch_per_gpu = 8  # local batch size — samples processed by each individual GPU

print("=== Distributed Data Parallel Simulation ===")
print(f"World size (GPUs):    {world_size}")
print(f"Batch per GPU:        {batch_per_gpu}")
print(f"Effective batch size: {batch_per_gpu * world_size}\n")

torch.manual_seed(42)
template = nn.Linear(8, 1, bias=False)  # shared initial model — all GPUs start with identical weights

gpu_grads = []  # collected gradients from each GPU — will be averaged via All-Reduce
for rank in range(world_size):  # rank: GPU identifier (0 to world_size-1), like a worker ID
    x = torch.randn(batch_per_gpu, 8)
    y = torch.randn(batch_per_gpu, 1)
    model = nn.Linear(8, 1, bias=False)
    model.weight.data.copy_(template.weight.data)
    loss = ((model(x) - y) ** 2).mean()
    loss.backward()
    gpu_grads.append(model.weight.grad.clone())
    print(f"  Rank {rank}: loss={loss.item():.4f}, grad_norm={model.weight.grad.norm():.4f}")

avg_grad = torch.stack(gpu_grads).mean(dim=0)  # synchronized gradient — identical on all GPUs after All-Reduce
print(f"\nAfter All-Reduce (gradient averaging):")
print(f"  Averaged grad norm: {avg_grad.norm():.4f}")
print(f"  All GPUs apply the SAME gradient -> models stay in sync")

# === What happens WITHOUT All-Reduce? Models diverge! ===
print(f"\nWithout sync (each GPU uses its own gradient):")
weights_diverged = []
for rank in range(world_size):
    w = template.weight.data.clone()
    w -= 0.01 * gpu_grads[rank]  # each GPU applies its OWN gradient
    weights_diverged.append(w)

w_synced = template.weight.data.clone() - 0.01 * avg_grad  # all apply SAME gradient

for rank in range(world_size):
    diff = (weights_diverged[rank] - w_synced).abs().mean().item()
    print(f"  Rank {rank} drift from synced model: {diff:.6f}")
print("Without sync, models drift apart -> training breaks!")

In [ ]:
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def train(rank, world_size):
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    model = GPT(config).cuda(rank)
    model = DDP(model, device_ids=[rank])
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, sampler=sampler, batch_size=32)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    for epoch in range(num_epochs):
        sampler.set_epoch(epoch)
        for batch in dataloader:
            loss = model(batch.cuda(rank)).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    dist.destroy_process_group()

---

**Course:** Aprenda LLM | **Chapter 10**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.